# Achtergrond & Biografie van Politici

**Thema:** Opleidingsachtergrond, loopbaan en demografie.

**Kernvragen:** Wat is de gemiddelde leeftijd, opleidingsachtergrond en geografische spreiding (geboorteplaats versus woonplaats) van politici? Welke werkgevers domineren hun eerdere loopbaan?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_onderwijs` en `persoon_loopbaan`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True) #connectie met de huidige database


def query_df(sql: str, parameters=None) -> pd.DataFrame:     #functie waarmee je sql kan schrijven
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

In [ ]:
persoondfdf= query_df("""select *
from silver.persoon""")
persoondfdf

## EDA
Met het dashboard worden de volgende vragen behandeld: Wat is de gemiddelde leeftijd, opleidingsachtergrond en geografische spreiding (geboorteplaats versus woonplaats) van politici? Welke werkgevers domineren hun eerdere loopbaan? En zijn deze vragen te categoriseren op fractie?


### Populatie

In [ ]:
import pandas as pd

#set met waarden die beschouwd worden als geen echte informatie 
PLACEHOLDERS = {"", "-", "onbekend", "n.v.t.", "nan", "none","geen opgave", "geen"}

#alles selecteren van persoonsgegevens
persoon_ruw = query_df("SELECT * FROM silver.persoon")


print(persoon_ruw["id"].is_unique) #true = geen dubbele ID's
print("ruw:", len(persoon_ruw)) #totaal aantal mensen
print(persoon_ruw["functie"].value_counts()) #telt hoevaak een waarde voorkomt

persoon = persoon_ruw
print("Rijen :", len(persoon))   


Aanname: een eerste kamerlid is ook een tweede kamerlid geweest omdat deze personen in de tweede kamer database voorkomen. Dashboard: Eerste Kamerleden apart tonen.

In [ ]:
# isna() mist placeholders; ze moeten als missing behandeld worden (PLACEHOLDERS, functie naar_na())
def naar_na(df):
    d = df.copy()
    for c in d.columns:
        if not (pd.api.types.is_numeric_dtype(d[c]) or pd.api.types.is_datetime64_any_dtype(d[c])): #api.types is return to pandas value
            leeg = d[c].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
            d[c] = d[c].mask(leeg, pd.NA)
    return d

persoon["geboortejaar"] = pd.to_datetime(persoon["geboortedatum"], errors="coerce").dt.year
persoon["decennium"] = persoon["geboortejaar"] // 10 * 10  #//= floordivision dus delen en naar beneden afronden

p = naar_na(persoon)
dekking = (p.drop(columns=["decennium"]).notna()
             .groupby(p["decennium"]).mean().mul(100).round(0))
dekking.insert(0, "n", p.groupby("decennium").size())
dekking   

In [ ]:
#kwaliteitsprofiel per tabel

def profiel(df, tabel): #dataframe + naam tabel
    n = len(df) #aantal rijen
    rijen = [] 
    for kol in df.columns:
        s = df[kol]
        nan = int(s.isna().sum()) #tellen aantal NAN
        plh = int((s.notna() & s.astype(str).str.strip().str.lower().isin(PLACEHOLDERS)).sum()) #waardes die nvt of - zijn niet nan
        rijen.append({"tabel": tabel, "kolom": kol, "n": n, "nan": nan,
                      "ingevuld_%": round(100 * (n - nan - plh) / n, 1), #totaal- nan- placeholders
                      "placeholder": plh, #aantal placeholders
                      "unieke": s.nunique()})
    return pd.DataFrame(rijen)

loopbaan  = query_df("SELECT * FROM silver.persoon_loopbaan")
onderwijs = query_df("SELECT * FROM silver.persoon_onderwijs")
loopbaan  = loopbaan[loopbaan["persoon_id"].isin(persoon["id"])].copy()
onderwijs = onderwijs[onderwijs["persoon_id"].isin(persoon["id"])].copy()

kwaliteit = pd.concat([profiel(persoon, "persoon"),  #pd.concat=samenvoegen van de tabellen gegeven door de functie
                       profiel(loopbaan, "persoon_loopbaan"),
                       profiel(onderwijs, "persoon_onderwijs")], ignore_index=True)
kwaliteit

In [ ]:
# hoeveel personen hebben zowel geboorteplaats als woonplaats echt ingevuld, en hoe zit het bij de huidige 150?


p = naar_na(persoon)      # placeholders ("-") worden hierin als leeg behandeld
dekking_functie = (p.groupby("functie")[["geboorteplaats", "woonplaats", "geboorteland"]]
                    .agg(lambda s: round(100 * s.notna().mean(), 1)))
dekking_functie.insert(0, "n", p.groupby("functie").size())
dekking_functie

- Populatie: 3083 personen (150 Tweede Kamerleden, 85 Eerste Kamerleden, 2848 oud-Kamerleden). 
- Geslacht en functie 100% ingevuld, een waarde is x: dat is ines kostic die is non binair, geboortedatum 99,9% (3 leeg). 
- Geboorteplaats 61,2% (30 NaN + 1166 keer "-")
- woonplaats 55,3%, geboorteland 21,3%, land 94,8% (160 NaN; vrijwel alleen NL, niet bruikbaar). 
- Overlijdensdatum 59,0% ingevuld; leeg is hier bij levende personen terecht, dus geen kwaliteitsmaat.\
**Per functie (geboorteplaats / woonplaats / geboorteland):**
 - Tweede Kamerlid 99,3 / 99,3 / 98,0; Eerste Kamerlid 69,4 / 88,2 / 62,4; 
 - oud-Kamerlid 59,0 / 52,0 / 16,1. Fractielabel is geen missing data: alleen partijleiders hebben het.

### Fractie
fractie_zetel_persoon  →  fractie_zetel  →  fractie
 (persoon + periode)       (de zetel)       (de partij)\
 \
 Deze join moet gemaakt worden om te kijken welke persoon er in welke fractie zit. Interessant om te kijken waar je een personen uit een fractie vandaan komen of hoe oud ze zijn.

In [ ]:
fractie = query_df("""
SELECT fzp.persoon_id, f.id AS fractie_id, f.afkorting, f.naam_nl,
       fzp.van, fzp.tot_en_met,
       ROW_NUMBER() OVER (PARTITION BY fzp.persoon_id
                          ORDER BY fzp.van ASC,  fzp.tot_en_met ASC  NULLS LAST)  AS rn_eerste,
       ROW_NUMBER() OVER (PARTITION BY fzp.persoon_id
                          ORDER BY fzp.van DESC, fzp.tot_en_met DESC NULLS FIRST) AS rn_laatste,
       COUNT(DISTINCT f.id) OVER (PARTITION BY fzp.persoon_id) AS n_fracties
FROM silver.fractie_zetel_persoon fzp
JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
JOIN silver.fractie f        ON f.id = fz.fractie_id
""")

kol = ["persoon_id", "fractie_id", "afkorting", "naam_nl", "van", "tot_en_met"]

eerste = (fractie[fractie["rn_eerste"] == 1][kol + ["n_fracties"]] #persoon bij de eerste fractie was
          .rename(columns={"fractie_id": "fractie_eerste_id", "afkorting": "fractie_eerste_afk",
                           "naam_nl": "fractie_eerste_naam", "van": "fractie_eerste_van",
                           "tot_en_met": "fractie_eerste_tot"})
          .set_index("persoon_id"))

laatste = (fractie[fractie["rn_laatste"] == 1][kol] #selecteren welke persoon bij laatste fractie was
           .rename(columns={"fractie_id": "fractie_laatste_id", "afkorting": "fractie_laatste_afk",
                            "naam_nl": "fractie_laatste_naam", "van": "fractie_laatste_van",
                            "tot_en_met": "fractie_laatste_tot"})
           .set_index("persoon_id"))

nieuw = list(eerste.columns) + list(laatste.columns)
persoon = persoon.drop(columns=[c for c in nieuw + ["persoon_id", "heeft_fractie", "start_jaar_fractie",
                                                    "partijwissel", "fractie_laatste_open"]
                                if c in persoon.columns])

n_voor = len(persoon)
persoon = (persoon.merge(eerste,  left_on="id", right_index=True, how="left")
                  .merge(laatste, left_on="id", right_index=True, how="left"))
assert len(persoon) == n_voor, "join heeft rijen verdubbeld"

persoon["heeft_fractie"]       = persoon["fractie_eerste_id"].notna()
persoon["start_jaar_fractie"]  = pd.to_datetime(persoon["fractie_eerste_van"], errors="coerce").dt.year
persoon["partijwissel"]        = persoon["n_fracties"] > 1  #persoon bij meerdere fracties heeft gezeten
persoon["fractie_laatste_open"] = persoon["heeft_fractie"] & persoon["fractie_laatste_tot"].isna()

print(persoon["heeft_fractie"].value_counts())    
print(persoon["partijwissel"].sum(), "personen met meer dan één fractie")

- van de totale populatie (3083) hebben er 803 personen een fractie koppeling
- Partij-visuals alleen voor de 803, met noemer; de rest heet "geen fractiekoppeling in de bron".

### Onderwijs
- Met REGEX (regular expressions) wordt er gezocht op trefwoorden in strings per record. Via deze strings kan er een categorie aan de record worden gegeven. Dus persoon A werkt als directeur bij een bank -> bedrijfsleven. 
- Belangrijk om te vermelden is dat deze methode een ruwe schatting geeft.
Een directeur bij de GGD moet ingedeeld worden op zorg maar komt helaas toch in bedrijfsleven terecht. Er is geprobeerd om deze fouten te voorkomen door middel van steekproeven. Dit garandeerd alleen geen honderd procent succes

In [ ]:
a=pd.DataFrame(loopbaan["werkgever"].unique())
a

In [ ]:
#opleiding categoriseren 
import re
import pandas as pd

PLACEHOLDERS |= {"geen opgave", "geen"}


def schoon(t):
    t = str(t).lower()
    t = re.sub(r"(?<=\w)\.(?=\w)", "", t)      # v.w.o. -> vwo, m.t.s. -> mts
    t = t.replace(".", "").replace('"', "")
    return re.sub(r"\s+", " ", t).strip()


def leeg(t):
    return pd.isna(t) or schoon(t) in PLACEHOLDERS

#met .unique alle unieke waardes gezocht en met ChatGPT laten indelen op categorie

NIET_VOLTOOID = r"niet voltooid|niet afgerond|onvoltooid|gestopt|afgebroken|\(\d+ jaar\)|propedeuse"

# Niveau op basis van de opleidingstekst. Volgorde telt: de eerste die matcht wint.
REGELS_TEKST = [
    ("cursus",
     r"cursus|certificaat|training|leergang|workshop|masterclass|"
     r"executive|postgraduate|postdoctoraal|postdoctoral|"
     r"\bnive\b|\bnima\b|coach|toezichthouder|mediation|"
     r"ondernemersvaardigheden|basis ?kwalificatie|register controller|"
     r"erasmusbeurs|vooropleiding|secretaresse|bestuurder|"
     r"management development|management trainee|bijscholing|"
     r"inservice|in-service|avondopleiding|avondcursus|"
     r"beroepsopleiding|kadertraining|"
     r"management course|leadership course|summer school|introductie|\bminor\b"),

    ("promotie",
     r"promotie|phd|gepromoveerd|doctorate|doctoraat"),

    ("hbo_expliciet",   # 'university of applied sciences' moet hbo zijn, niet wo
     r"applied sciences|hoger beroepsonderwijs|higher vocational"),

    ("wo",
     r"master|msc|\bllm\b|doctoraal|\bdrs\b|\bir\b|ingenieur|"
     r"bachelor|\bba\b|\bbsc\b|\bma\b|\bmba\b|\bmphil\b|"
     r"kandidaats|universitair|\bwo\b|universitaire|university|doctorandus"),

    ("hbo",
     r"\bhbo\b|hogeschool|pabo|\bhts\b|\bheao\b|\bhes\b|"
     r"lerarenopleiding|kunstacademie|hotelschool"),

    ("mbo",
     r"\bk?mbo\b|\bmts\b|meao|middelbaar beroepsonderwijs|"
     r"ambachtsschool|\bito\b|kleuterleidster|verpleegkundige|"
     r"spd[- ]bedrijfsadministratie|vakopleiding"),

    ("vo",
     r"mavo|havo|vwo|gymnasium|atheneum|\bhbs\b|mulo|lyceum|"
     r"vmbo|\blbo\b|\bulo\b|middelbare|vbo|leao|mms|"
     r"algemeen voortgezet onderwijs|"
     r"secondary school|secondary education|highschool|high school"),

    ("basisonderwijs",
     r"basisschool|lagere school|primary school|primary education"),
]

# Niveau op basis van de instelling (terugval). Volgorde telt.
INSTELLING_REGELS = [
    ("hbo",
     r"hogeschool|hogescholen|applied sciences|\bhbo\b|\bheao\b|\bhes\b|"
     r"fontys|saxion|windesheim|avans|inholland|\bhva\b|\bhu\b|hanze|"
     r"academie van bouwkunst|hotelschool|militaire academie"),

    ("wo",
     r"universiteit|universitair|university|\buva\b|\bvu\b|\btu\b|"
     r"erasmus|radboud|nyenrode|\bkub\b|\brug\b|"
     r"london school|school of economics|sorbonne|oxford|\beur\b"),

    ("mbo",
     r"\broc\b|\bmbo\b|koning willem|albeda"),

    ("vo",
     r"gymnasium|lyceum|\bmavo\b|\bhavo\b|\bvwo\b|\bvbo\b|\bleao\b|"
     r"scholengemeenschap|college\b|middelbare school|secondary school"),

    ("basisonderwijs",
     r"basisschool|primary school|lagere school|primary education"),

    ("cursus",
     r"ministerie|orde van advocaten|vereniging|stichting|"
     r"psychodynamica|schoevers|sioo"),
]


def niveau_instelling(instelling):
    if leeg(instelling):
        return None
    i = schoon(instelling)
    for naam, pat in INSTELLING_REGELS:
        if re.search(pat, i):
            return naam
    return None


def niveau(opl, instelling):
    """Geeft (categorie, bron, status). bron = waar het niveau vandaan komt."""
    status = "voltooid"
    if not leeg(opl) and re.search(NIET_VOLTOOID, schoon(opl)):
        status = "niet_voltooid"

    if not leeg(opl):
        t = schoon(opl)
        for naam, pat in REGELS_TEKST:
            if re.search(pat, t):
                return ("hbo" if naam == "hbo_expliciet" else naam), "tekst", status

    ni = niveau_instelling(instelling)          # terugval: soort instelling, niet de graad
    if ni is not None:
        return ni, "instelling", status

    return ("onbekend" if leeg(opl) else "niet_ingedeeld"), "geen", status


CANON = [
    (r"universiteit van amsterdam|\buva\b", "Universiteit van Amsterdam"),
    (r"vrije universiteit(?! brussel)|\bvu\b", "Vrije Universiteit Amsterdam"),
    (r"leiden",                             "Universiteit Leiden"),
    (r"erasmus",                            "Erasmus Universiteit Rotterdam"),
    (r"groningen",                          "Rijksuniversiteit Groningen"),
    (r"utrecht",                            "Universiteit Utrecht"),
    (r"radboud|nijmegen",                   "Radboud Universiteit"),
    (r"tilburg|brabant",                    "Tilburg University"),
    (r"maastricht",                         "Universiteit Maastricht"),
    (r"twente",                             "Universiteit Twente"),
    (r"wageningen",                         "Wageningen University"),
    (r"delft",                              "TU Delft"),
    (r"nyenrode",                           "Nyenrode Business Universiteit"),
    (r"open universiteit",                  "Open Universiteit"),
    (r"oxford",                             "University of Oxford"),
]


def instelling_schoon(x):
    if leeg(x):
        return pd.NA
    if niveau_instelling(x) != "wo":
        return str(x).strip()
    s = schoon(x)
    for pat, naam in CANON:
        if re.search(pat, s):
            return naam
    return str(x).strip()


# toepassen op elke opleidingsrij van de onderwijstabel
onderwijs = onderwijs.drop(columns=[c for c in ["categorie", "bron", "status", "rang", "instelling_schoon"]
                                    if c in onderwijs.columns])
onderwijs[["categorie", "bron", "status"]] = onderwijs.apply(
    lambda r: niveau(r["opleiding_nl"], r["instelling"]), axis=1, result_type="expand")
onderwijs["instelling_schoon"] = onderwijs["instelling"].map(instelling_schoon)


# hoogste voltooide niveau per persoon
RANG = {"basisonderwijs": 0, "vo": 1, "mbo": 2, "hbo": 3, "wo": 4, "promotie": 5}  #dictionary
onderwijs["rang"] = onderwijs["categorie"].map(RANG)
geldig = onderwijs[(onderwijs["status"] == "voltooid") & onderwijs["rang"].notna()].copy()
# bij gelijke rang wint 'tekst' boven 'instelling' (True sorteert laatst, tail(1) pakt die)
geldig["bron_is_tekst"] = geldig["bron"] == "tekst"
hoogste = (geldig.sort_values(["rang", "bron_is_tekst"]).groupby("persoon_id").tail(1)
           [["persoon_id", "categorie", "bron"]]
           .rename(columns={"categorie": "opleiding_hoogste", "bron": "opleiding_hoogste_bron"})
           .set_index("persoon_id"))
n_opl = onderwijs.groupby("persoon_id").size().rename("n_opleidingen")

persoon = persoon.drop(columns=[c for c in ["opleiding_hoogste", "opleiding_hoogste_bron",
                                            "n_opleidingen", "opleiding_status"] if c in persoon.columns])
n_voor = len(persoon)
persoon = (persoon.merge(hoogste, left_on="id", right_index=True, how="left")
                  .merge(n_opl,   left_on="id", right_index=True, how="left"))
assert len(persoon) == n_voor, "merge heeft personen verdubbeld"
persoon["n_opleidingen"] = persoon["n_opleidingen"].fillna(0).astype(int)

persoon["opleiding_status"] = "geen opleiding geregistreerd"
persoon.loc[persoon["n_opleidingen"] > 0, "opleiding_status"] = "geregistreerd, geen voltooid niveau af te leiden"
persoon.loc[persoon["opleiding_hoogste"].notna(), "opleiding_status"] = "niveau bepaald"

# wat is ingedeeld en wat niet (per opleidingsrij)
onderwijs["ingedeeld"] = ~onderwijs["categorie"].isin(["niet_ingedeeld", "onbekend"])
print(onderwijs["ingedeeld"].value_counts())
print(onderwijs["categorie"].value_counts())

# wat het dashboard gebruikt (per persoon)
print(persoon["opleiding_status"].value_counts())
print(persoon["opleiding_hoogste"].value_counts())

print(persoon["opleiding_hoogste_bron"].value_counts())


Van de 979 opleidingsrijen zijn er 849 (86,7%) ingedeeld op niveau; 130 (13,3%) zijn niet ingedeeld of onbekend. Het hoogste voltooide niveau is bepaald voor 265 van de 3083 personen (8,6%): wo 193, hbo 30, vo 19, promotie 15, mbo 8. Bij 110 van die 265 (41,5%) is het niveau geschat op basis van de instelling, bij 155 komt het uit de opleidingstekst. 13 personen hebben opleidingsrijen maar geen af te leiden voltooid niveau, en 2805 hebben geen opleidingsrij. "Geen opleiding geregistreerd" is niet hetzelfde als "geen opleiding gevolgd". Cursussen en niet voltooide opleidingen tellen niet mee voor het hoogste niveau.

In [ ]:
print(pd.crosstab(persoon["decennium"], persoon["opleiding_status"], normalize="index").round(2).to_string())
print(pd.crosstab(persoon["functie"], persoon["opleiding_status"]).to_string())

Relevante populatie voor opleiding: personen met een fractiekoppeling, geboren vanaf 1950 (n = 735). Daarvan heeft 262 (35,6%) een bepaald niveau, 460 (62,6%) geen opleiding geregistreerd en 13 geen af te leiden niveau.

In [ ]:
# 1. aantallen erbij, alleen vanaf 1940
t = pd.crosstab(persoon["decennium"], persoon["opleiding_status"])
t["n"] = t.sum(axis=1)
t["bepaald_%"] = (100 * t["niveau bepaald"] / t["n"]).round(0)
print(t.loc[1940:].to_string())

# 2. hangt opleiding samen met de fractiedata? (dezelfde bron-periode?)
print(pd.crosstab(persoon["heeft_fractie"], persoon["opleiding_status"]).to_string())

# 3. alleen de 150 Tweede Kamerleden: wanneer zijn ze begonnen, bij welke fractie?
tk = persoon[persoon["functie"] == "Tweede Kamerlid"].copy()
tk["start_bin"] = pd.cut(tk["start_jaar_fractie"], [1976, 2006, 2012, 2017, 2021, 2023, 2026])
print(pd.crosstab(tk["start_bin"], tk["opleiding_status"]).to_string())
print(pd.crosstab(tk["fractie_eerste_afk"], tk["opleiding_status"]).to_string())

### Loopbaan
Ook hier is REGEX een prima ruwe inschatting maar niet foutloos. Een directeur van safe the children wordt ingedeeld op bedrijfsleven terwijl een NGO (non governmental organisation) eigenlijk maatschappelijk is. Daarom wordt er ingedeeld op "bedrijfsleven en overige organisaties "

In [ ]:
loopbaan["functie"]   = loopbaan["functie"].fillna("")
loopbaan["werkgever"] = loopbaan["werkgever"].fillna("")
loopbaan["tekst"] = (loopbaan["functie"] + " | " + loopbaan["werkgever"]).str.lower()

POLITIEK_CONTEXT = (r"fractie|tweede kamer|eerste kamer|kamerlid|partij|campagne|verkiezing|wetenschappelijk bureau|"
                    r"\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu|sgp|volt|denk|bbb|fvd)\b")



REGELS_SECTOR = [
    (
        "kamerlidmaatschap",
        r"kamerlid|tweede kamer(?: der staten-generaal)?|eerste kamer(?: der staten-generaal)?|senaatslid",
    ),
    (
        "politiek_ondersteuning",
        r"medewerk(?:er|ster)?|campagne|(?<!staats)secretaris|voorlichter|fractiebureau|kamerfractie|"
        r"woordvoerder|assistent|politiek(?:e)? (?:adviseur|assistent)|beleidsadviseur|"
        r"co[oö]rdinerend beleidsadviseur|\bbeleid\b|policy officer|politiek(?:e)? co[oö]rdinat|speechschrijver|"
        r"verkiezingsprogramma|parlementaire (?:redactie|en departementale contacten)|"
        r"advies(?:eur|ering).*(?:minister|parlement|fractie|verkiez|coalitie|kabinets|politiek|binnenlands bestuur)|"
        r"wetenschappelijk bureau (?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|cu)|"
        r"fondsenwerver.*\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu)\b",
    ),
    (
        "politiek_bestuur",
        r"fractievoorzitter|lijsttrekker|parlementslid|informateur|formateur|europees parlement|europarlementari(?:ë|e)r|"
        r"comit[eé] van de regio|vaste commissie|\blid db\b|\blid ab\b|"
        r"\bminister\b|minister-president|staatssecretaris|wethouder|burgemeester|"
        r"gemeenteraad|provinciale staten|gedeputeerde|raadslid|statenlid|statenfractie|"
        r"stadsdeelraad|stadsdeelvoorzitter|burgerstatenlid|duo-statenlid|"
        r"steunfractielid|fractievertegenwoordiger|bestuurscommissielid|partij|"
        r"\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu|sgp|volt|denk|bbb|"
        r"forum voor democratie|fvd|partij voor de dieren)\b|cdja|christen democratisch jongeren|jonge democraten|jovd",
    ),
    (
        "media_communicatie",
        r"journalist|verslaggever|redacteur|hoofdredacteur|columnist|presentat|programmamaker|"
        r"omroep|televisie|\bradio\b|\bavro\b|\bnos\b|\bncrv\b|persvoorlichter|"
        r"nieuwslezer|redactielid|radio(?:columnist|culumnist)|documentairemaker|documentaire|toneel(?:makerij)?|actrice|acteur|voice[ -]?over|\bdj\b|"
        r"producent|publicist|tekstschrijver|mediabedrijf|communicatie\s*(?:adviseur|advies|manager|co[oö]rdinat)|"
        r"hoofd (?:corporate )?communicatie|hoofd informatie.*communicatie|hoofd voorlichting|\bvoorlichting\b",
    ),
    (
        "juridisch",
        r"advocaat|rechter|jurist|notaris|officier van justitie|fiscalist|belastingadviseur|"
        r"(?:company|legal) lawyer|legal advisor|bewindvoerder|mediator|mediation|procesvertegenwoordiger|"
        r"griffier|raadsheer|zittingsvertegenwoordiger|rechtstoepassing|personenschade|vennootschapsbelasting|(?:aanbestedings|staats)recht|"
        r"wetgevings|wet mulder|habeas corpus|formeel recht",
    ),
    (
        "onderwijs_wetenschap",
        r"universiteit|university|hogeschool|docent|leraar|hoogleraar|professor|onderwijs|school|"
        r"onderzoek|\bcito\b|examen|vmbo|\bvwo\b|\bhavo\b|\bhbo\b|postdoctoral|postdoc|"
        r"promovend|promotietraject|proefschrift|archeoloog|\blector\b|\brector\b|cursusleider|"
        r"afdelingsleider|stagebegeleider|studentadviseur|taalles|propedeuse|opleiding|\bects\b",
    ),
    (
        "zorg",
        r"\barts\b|ziekenhuis|verpleeg|zorg|huisarts|ggz|verloskund|laborant|pati[eë]nt|oncologie|"
        r"medicatieveiligheid|jeugdhulp|gezinsvoogd|jeugdwelzijn|maatschappelijk werker|"
        r"re[- ]integratie|gedragswetenschapper|beschermd wonen|preventieakkoord|\bggd\b|gezondheidsdienst",
    ),
    (
        "overheid",
        r"ministerie|\bgemeente|provincie|rijks|overheid|waterschap|ambassadeur|europese commissie|"
        r"belastingdienst|\bvng\b|ambtenaar|handhaver|belastingtelefoon|toeslagen|behandelfunctionaris|rekenkamer(?:commissie)?|"
        r"diplomaat|diplomatie|gezant|zaakgelastig|permanente vertegenwoordiger|"
        r"vertegenwoordiger van nederland|democratie en bestuur|inwonerszaken|"
        r"sociale zaken en werkgelegenheid|consulent wet inschakeling werkzoekenden|afdelingshoofd.*milieu|stafbureau|grondgebied|c2000|veiligheidsregio|volksraadpleging|openbare orde|"
        r"kabinetsaanpak|begrotingsbeleid|klantdirectie|stedelijke vernieuwing|decentralisat|democratisering|staatsadviescommissie|verkeer en vervoer|concentratiecontrole|planeconoom",
    ),
    (
        "maatschappelijk",
        r"vakbond|\bfnv\b|\bcnv\b|stichting|vereniging|belangen|\bbond\b|politiebond|"
        r"jeugdraad|oxfam|rotary|abvakabo|partos|diaken|ouderling|kerk|geloof|theologi|"
        r"vrijwillig|mensenrechten|rechten van de mens|opbouwwerk|sociaal[- ]cultureel|sociaal raadsvrouw|jeugdcommissie|"
        r"oudercommissie|leerlingenraad|studentenplatform|stop kindermisbruik",
    ),
    (
        "veiligheid_defensie",
        r"vlieg|squadron|\bf-16\b|\bpolitie\b|politieagent|militair|defensie|marine|luchtmacht|"
        r"marechaussee|rechercheur|hoof?dinspecteur|commandant|peloton|sergeant|officier|"
        r"legerkorps|reserve[- ]?officier|panzer|vuurmond|geleide wapens|vluchtcommandant|"
        r"wing scheduler|\bt-38\b|rsuto|runway supervisor|inlichtingen(?:-| )en veiligheids|veiligheidsvraagstuk",
    ),
    (
        "bedrijfsleven en overige organisaties",
        r"\bb\.?v\.?\b|\bn\.?v\.?\b|\bv\.?o\.?f\b|directeur|director|manager|management|"
        r"consultant|bank|verzekering|eigenaar|ondernemer|oprichter|vice president|controller|account|"
        r"marketing|sales|public affairs|\bhead of\b|personeelszaken|tester|adviesbureau|\bey\b|"
        r"ernst & young|deloitte|\bkpn\b|\b(?:cfo|ceo|dga)\b|chief economist|business|commercieel|"
        r"financieel|fixed income|kredietcrisis|investment|economist|analist|analyst|\bhr\b|\bp&o\b|"
        r"personeel|\bit(?:-er)?\b|\bict\b|systeem(?:beheer|ontwerp)|software|sysadmin|engineer|"
        r"constructeur|architect|bouw(?:techniek|kund|fysica)|vastgoed|technisch|kwaliteit|arbo|inkoop|"
        r"logistiek|bedrijfs(?:leider|voering|bureau)|industrie|retail|horeca|landbouw|melkvee|"
        r"boulanger|patissier|serveerster|chauffeur|machinist|verkoper|acquisiteur|reisbegeleid|"
        r"dier(?:enarts|enartsen)|consult|zzp|zelfstandig|freelance|\bowner\b|\bpartner\b|"
        r"treasurer|penningmeester|\btrainee\b|programmeur|projectontwikkelaar|procesontwikkeling|"
        r"ontwerper|teamleider|co[oö]rdinator|projectleider|programmaco[oö]rdinator|programmaleider|"
        r"directielid|directievoorzitter|\braad van bestuur\b|\braad van commissarissen\b|"
        r"\braad van toezicht\b|\bondernemingsraad\b|\badviseur\b|organisatie[- ]?adviseur|projectadviseur|bedrijvenadviseur|huisvestingsadviseur|subsidieadviseur|bedrijfsleid(?:er|ster)|leidinggevende|\bhead\b|senior associate|\btrainer\b|\bcoach\b|secretaresse|archivaresse|systeem[-/ ]*(?:netwerk)?beheer(?:der)?|quality inspector|\beigenares\b|\becono(?:om|misch)\b|\bagriculture\b|\bagrarisch\b|\bbouw\b|stafadviseur|onderhandelaar|consulent|clusterleider|groepsleider|vestigingsleider|coördinatrice|r&d|informatievoorziening|automatisering|\bexpert\b|\binterim\b",
    ),
    (
        "bestuur_toezicht_onvoldoende_sectorcontext",
        r"bestuurder|bestuurslid|\bbestuur\b|voorzitterr?|vice[- ]?voorzitter|commissaris|"
        r"\blid\b|adviesraad|adviesforum|auditcommissie|klankbordgroep|werkgroep|kernteam|jurylid|"
        r"panellid|presidium|portefeuillehouder|toezichtraad|commissielid|raad van advies|president|bestuurlijk",
    ),
    ("overig_onvoldoende_sectorcontext", r".+"), #telt als buiten politiek werk
]

STAF_IN_FUNCTIE = r"medewerk|adviseur|assistent|secretaris|voorlichter|woordvoerder|\bbeleid|speechschrijver|stagiair|campagne"
ECHT_KAMERLID   = r"\bkamerlid\b|\blid\b.*\bkamer\b|senaatslid"

def sector(tekst):
    if not tekst.strip(" |"):
        return "onbekend"
    functie = tekst.split(" | ")[0]
    for naam, patroon in REGELS_SECTOR:
        if re.search(patroon, tekst):
            if naam == "politiek_ondersteuning" and not re.search(POLITIEK_CONTEXT, tekst):
                continue
            if naam == "kamerlidmaatschap" and re.search(STAF_IN_FUNCTIE, functie) \
               and not re.search(ECHT_KAMERLID, functie):
                continue          # staf van een fractie is geen Kamerlid
            return naam
    return "niet_ingedeeld"

uniek_l = loopbaan[["tekst"]].drop_duplicates().copy()
uniek_l["sector"] = uniek_l["tekst"].map(sector)
loopbaan["sector"] = loopbaan["tekst"].map(sector)

print(uniek_l["sector"].value_counts())
uniek_l[uniek_l["sector"] == "niet_ingedeeld"].head(60)     # hier verbeter je je regels


import re
BINNEN_POLITIEK = {"kamerlidmaatschap", "politiek_ondersteuning", "politiek_bestuur"}
ONDUIDELIJK = {"overig_onvoldoende_sectorcontext", "bestuur_toezicht_onvoldoende_sectorcontext"}
loopbaan["buiten_politiek"] = ~loopbaan["sector"].isin(BINNEN_POLITIEK | {"onbekend", "niet_ingedeeld"})
loopbaan["buiten_zeker"]    = ~loopbaan["sector"].isin(BINNEN_POLITIEK | ONDUIDELIJK | {"onbekend", "niet_ingedeeld"})

lb = loopbaan.groupby("persoon_id").agg(
    n_loopbaanrijen=("sector", "size"),
    n_buiten_politiek=("buiten_politiek", "sum"),
    n_buiten_zeker=("buiten_zeker", "sum"),
)

kol = ["n_loopbaanrijen", "n_buiten_politiek", "n_buiten_zeker"]
persoon = persoon.drop(columns=[c for c in kol if c in persoon.columns])
n_voor = len(persoon)
persoon = persoon.merge(lb, left_on="id", right_index=True, how="left")
assert len(persoon) == n_voor, "merge heeft personen verdubbeld"
persoon[kol] = persoon[kol].fillna(0).astype(int)

m = persoon[persoon["n_loopbaanrijen"] > 0]
print("personen met loopbaan:", len(m))
print("ondergrens (zeker buiten politiek):", (m["n_buiten_zeker"] > 0).sum(), f"= {100*(m['n_buiten_zeker'] > 0).mean():.1f}%")
print("bovengrens (incl. onduidelijk):    ", (m["n_buiten_politiek"] > 0).sum(), f"= {100*(m['n_buiten_politiek'] > 0).mean():.1f}%")
print(persoon.groupby("functie")["n_loopbaanrijen"].apply(lambda s: round(100 * (s == 0).mean(), 1)))

print("geen loopbaanrijen:", round(100 * (persoon["n_loopbaanrijen"] == 0).mean(), 1), "%")
print("geen loopbaan buiten politiek:", round(100 * (persoon["n_buiten_politiek"] == 0).mean(), 1), "%")
print(pd.crosstab(persoon["heeft_fractie"], persoon["n_loopbaanrijen"] > 0, margins=True).to_string())

f = persoon[persoon["heeft_fractie"] & (persoon["geboortejaar"] >= 1950)]
print(len(f))
print(f["opleiding_status"].value_counts().to_string())

print(pd.crosstab(persoon["heeft_fractie"], persoon["n_loopbaanrijen"] > 0, margins=True).to_string())




Loopbaanrijen bestaan voor 278 van de 3083 personen (9,0%), allemaal binnen de 803 met fractiekoppeling. Van de huidige Tweede Kamerleden heeft 62,0% (93 van 150) geen enkele loopbaanrij; bij Eerste Kamerleden is dat 96,5% en bij oud-Kamerleden 92,3%. Dat is niet hetzelfde als "geen baan vóór het Kamerlidmaatschap": dat is niet getest. Van de 278 met loopbaan heeft 94,6% (263) zeker en 96,8% (269) hooguit minstens één baan buiten de politiek, afhankelijk van of de onduidelijke sectoren meetellen. De indeling is regelgebaseerd en op 50 unieke teksten door mij gecontroleerd.

In [ ]:
# 1. zijn het dezelfde 275 personen?
print(pd.crosstab(persoon["n_opleidingen"] > 0, persoon["n_loopbaanrijen"] > 0, margins=True).to_string())

# 2. jouw 62%: opnieuw berekend voor de huidige Tweede Kamerleden
tk = persoon[persoon["functie"] == "Tweede Kamerlid"]
print(len(tk), round(100 * (tk["n_loopbaanrijen"] == 0).mean(), 1), " % heeft geen enkele loopbaanrij geregistreerd.")

# 3. onder de 275 met loopbaan: hoeveel hebben iets buiten de politiek?
m = persoon[persoon["n_loopbaanrijen"] > 0]
print(len(m), round(100 * (m["n_buiten_politiek"] > 0).mean(), 1), "% heeft minstens één baan buiten politiek")

In [ ]:
n_rijen = loopbaan["tekst"].value_counts().rename("n_rijen")
uniek_l = uniek_l.merge(n_rijen, left_on="tekst", right_index=True, how="left")
uniek_l[uniek_l["sector"] == "niet_ingedeeld"].sort_values("n_rijen", ascending=False).head(60)

In [ ]:
persoon["loopbaan_status"] = "geen loopbaan geregistreerd"
persoon.loc[persoon["n_loopbaanrijen"] > 0, "loopbaan_status"] = "geregistreerd, geen baan buiten politiek ingedeeld"
persoon.loc[persoon["n_buiten_politiek"] > 0, "loopbaan_status"] = "geregistreerd, baan buiten politiek"
print(persoon["loopbaan_status"].value_counts().to_string())

Loopbaangegevens zijn geregistreerd voor 275 van de 2998 personen (9,2%), vrijwel dezelfde personen als bij opleiding (272 overlap). Bij de 150 huidige Tweede Kamerleden heeft 62% geen enkele loopbaanrij. Van de 275 met loopbaan heeft 87,6% minstens één functie buiten politiek; 21% van de unieke teksten kon niet worden ingedeeld. De oorzaak van de ontbrekende gegevens is niet af te leiden uit de data.

De populatie is 2998 personen: 150 huidige Tweede Kamerleden en 2848 oud-Kamerleden.\
Deze populatie bestaat weer uit twee lagen:\
Alle personen	2998	geslacht (100%), geboortedatum (99,9%), geboorteplaats (61%), woonplaats (54%), overlijdensdatum (61%)\
"Rijke" laag	791 (26%)	fractie, portret, en daarbinnen 275 met opleiding én loopbaan (9%)\


In [ ]:
# zijn de 791 'rijke' records dezelfde personen?
print(pd.crosstab(persoon["portret_url"].notna(), persoon["heeft_fractie"]))

# wat zijn die 1166 placeholders, en wat is 'land'?
print(persoon["geboorteplaats"].value_counts().head(10))
print(persoon["land"].value_counts())

# hoeveel personen hebben zowel geboorteplaats als woonplaats echt ingevuld, en hoe zit het bij de huidige 150?
p = naar_na(persoon)
beide = p["geboorteplaats"].notna() & p["woonplaats"].notna()
print(beide.sum(), round(100 * beide.mean(), 1), "%")
print(p.groupby("functie")[["geboorteplaats", "woonplaats", "geboorteland"]]
        .agg(lambda s: round(100 * s.notna().mean(), 1)))

In [ ]:
afwijkend = persoon[persoon["portret_url"].notna() & ~persoon["heeft_fractie"]]
print(afwijkend[["achternaam", "voornamen", "functie", "geboortejaar", "n_loopbaanrijen", "n_opleidingen"]].to_string())

In [ ]:
ids = afwijkend["id"].tolist()
lijst = ", ".join(f"'{i}'" for i in ids)
tabellen = ["activiteit_actor", "commissie_zetel_vast_persoon", "commissie_zetel_vervanger_persoon",
            "document_actor", "zaak_actor", "stemming"]
rijen = []
for t in tabellen:
    r = query_df(f"SELECT COUNT(*) AS n, COUNT(DISTINCT persoon_id) AS personen FROM silver.{t} WHERE persoon_id IN ({lijst})")
    rijen.append((t, int(r['n'][0]), int(r['personen'][0])))
pd.DataFrame(rijen, columns=["tabel", "rijen", "personen_van_de_10"])

* De populatie is 2998 personen: 150 huidige Tweede Kamerleden en 2848 oud-Kamerleden.
* Geslacht is 100% ingevuld, geboortedatum 99,9%. Daarna loopt de dekking sterk terug: geboorteplaats 61% (waarvan 1166 keer "-"), woonplaats 54%, geboorteland 20%.
* De huidige 150 zijn bijna compleet voor geboorteplaats, woonplaats en geboorteland (98 tot 99%).    Bij oud-Kamerleden is dat 59%, 52% en 16%.
* Fractie bestaat voor 791 personen (26% van totaal). Alle 275 personen met opleiding en loopbaan zitten daarin (272 overlap).
* Opleidingsniveau is bepaald voor 263 personen, van wie 128 geschat uit de instelling. Bij de huidige Kamer heeft 56 van 150 een niveau.
* 93 van de 150 huidige Kamerleden hebben geen enkele loopbaanrij.
* land is bijna constant (NL) en niet bruikbaar.
\
Niet te beantwoorden uit de data: waarom opleiding en loopbaan bij maar 9% staan.

### Steekproef

Met de steekproef is er gekeken naar de kwaliteit van de indelingscode en met kritiek past Claude de REGEX code aan. 

In [ ]:
# onderwijs
s1 = onderwijs.drop_duplicates(["opleiding_nl", "instelling"]).sample(50, random_state=2)
s1[["opleiding_nl", "instelling", "categorie", "bron"]].assign(juist="").to_csv("check_onderwijs.csv", index=False)

# loopbaan
s2 = loopbaan.drop_duplicates("tekst").sample(50, random_state=2)
s2[["tekst", "sector"]].assign(juist="").to_csv("check_loopbaan.csv", index=False)

s1

steekproef van 50 unieke teksten per tabel handmatig beoordeeld: onderwijs ~90% van de ingedeelde teksten onomstreden juist, loopbaan buiten_politiek 92% juist, sectorlabel 74% juist. Beoordeling door één persoon."

### Handmatige correctie
- Voor mevrouw  I. (Iem) al Biyati ontbrak de geboortedatum als enige van het huidig zittende parlement. Daardoor is er ook verder gekeken naar of ze een loopbaan heeft aangegeven en opleidingen. Dit staat ook niet in de database. Om de data voor het huidige parlement aan te vullen is de geboortedatum en geboorteplaats aangevuld. Loopbaan en onderwijs zijn niet nagelopen omdat ze dan anders wordt behandeld dan de andere 149 kamerleden die niet individueel zijn nagelopen.
- Dit geval doet wel vermoeden dat de database achterloopt op de huidige starters in het parlement. Dit is een aanwijzing geen bewijs

In [ ]:
pid = "01be9ad7-0444-483a-9983-5f66f1402f6a"

# stap 0: hulpkolommen altijd (opnieuw) aanmaken, dan staat de volgorde niet meer in de weg
persoon["geboorte_d"]   = pd.to_datetime(persoon["geboortedatum"], errors="coerce")
persoon["overlijden_d"] = pd.to_datetime(persoon["overlijdensdatum"], errors="coerce")
persoon["geboortejaar"] = persoon["geboorte_d"].dt.year
persoon["geboorteplaats"] = persoon["geboorteplaats"].astype("object")

m = persoon["id"] == pid
assert m.sum() == 1, f"verwacht 1 rij, gevonden {m.sum()}"

# 1. wat staat er in de bron? (alleen kolommen die bestaan)
kol = ["functie", "geboortedatum", "geboorteplaats", "woonplaats", "opleiding_status",
       "opleiding_hoogste", "n_loopbaanrijen", "n_buiten_zeker"]
print("ontbrekende kolommen:", [c for c in kol if c not in persoon.columns])
print(persoon.loc[m, [c for c in kol if c in persoon.columns]].to_string())

# 2. correctie vastleggen (alleen invullen wat in de bron leeg is)
CORRECTIES = {
    pid: {"geboortedatum": "1998-07-12", "geboorteplaats": "Amsterdam",
          "bron": "parlement.com", "opgezocht": "2026-10-08"},
}
persoon["handmatig_aangevuld"] = persoon["id"].isin(CORRECTIES)
for k, c in CORRECTIES.items():
    m = persoon["id"] == k
    assert m.sum() == 1
    d = pd.to_datetime(c["geboortedatum"])
    if persoon.loc[m, "geboorte_d"].isna().all():
        persoon.loc[m, "geboorte_d"] = d
        persoon.loc[m, "geboortejaar"] = d.year
    if pd.isna(persoon.loc[m, "geboorteplaats"].iloc[0]):
        persoon.loc[m, "geboorteplaats"] = c["geboorteplaats"]
    if persoon.loc[m, "geboortedatum"].isna().all():
        persoon.loc[m, "geboortedatum"] = c["geboortedatum"]

print(persoon.loc[m, ["geboorte_d", "geboortejaar", "geboorteplaats", "handmatig_aangevuld"]].to_string())

### Demografie

In [ ]:
REFDATUM = pd.Timestamp("2026-10-08")      # vaste datum, zodat de uitkomst reproduceerbaar is
persoon["geboorte_d"] = pd.to_datetime(persoon["geboortedatum"], errors="coerce")
persoon["overlijden_d"] = pd.to_datetime(persoon["overlijdensdatum"], errors="coerce")
persoon["geboortejaar"] = persoon["geboorte_d"].dt.year

# 1. Hoe volledig is de overlijdensdatum? (geboren <= 1900 is zeker overleden)
dec = persoon["geboortejaar"] // 10 * 10
print(persoon.groupby(dec)["overlijden_d"].apply(lambda s: round(s.notna().mean(), 2)).to_string())

# 2. Foutcontrole
sterfleeftijd = (persoon["overlijden_d"] - persoon["geboorte_d"]).dt.days / 365.25
print("overlijden voor geboorte:", (sterfleeftijd < 0).sum(), "| >110 jaar:", (sterfleeftijd > 110).sum(),
      "| overlijden na refdatum:", (persoon["overlijden_d"] > REFDATUM).sum())

# 3. De 150 huidige Tweede Kamerleden leven per definitie: hoeveel hebben toch een overlijdensdatum?
print(persoon.loc[persoon["functie"] == "Tweede Kamerlid", "overlijden_d"].notna().sum())


def levensstatus(df, drempel):
    s = pd.Series("onbekend (oud, geen overlijdensdatum)", index=df.index)
    s[df["overlijden_d"].notna()] = "overleden (datum bekend)"
    s[df["overlijden_d"].isna() & (df["geboortejaar"] >= drempel)] = "aangenomen in leven"
    s[df["overlijden_d"].isna() & df["geboortejaar"].isna()] = "onbekend (geen geboortedatum)"
    return s

for d in (1915, 1920, 1925, 1930):
    print(d, levensstatus(persoon, d).value_counts().to_dict())

In [ ]:
# steekproef voor Parlement.com
kol = [c for c in ["id", "achternaam", "roepnaam", "geboortedatum", "functie"] if c in persoon.columns]
check = persoon[persoon["overlijden_d"].isna() & persoon["geboortejaar"].between(1930, 1949)]
print(len(check))
print(check.sample(10, random_state=1)[kol].to_string())

# leeftijd bij aantreden (803 met fractie); alleen jaarverschil, dus ±1 jaar
persoon["leeftijd_aantreden"] = persoon["start_jaar_fractie"] - persoon["geboortejaar"]
a = persoon["leeftijd_aantreden"].dropna()
print(len(a)); print(a.describe().round(1).to_string())
print("onwaarschijnlijk (<18 of >80):", ((a < 18) | (a > 80)).sum())

# huidige leeftijd, alleen de 150 huidige Tweede Kamerleden
tk = persoon[persoon["functie"] == "Tweede Kamerlid"].copy()
tk["leeftijd_nu"] = ((REFDATUM - tk["geboorte_d"]).dt.days / 365.25).round(1)
print(tk["leeftijd_nu"].describe().round(1).to_string())

- De tien namen zijn gecontroleerd op parlement.com . Er is geen overlijdensdatum opgegeven voor deze personen waardoor de aanname "geen overlijdensdatum= nog in leven". Wél wordt dit alleen aangenomen voor personen na 1930. Iemand uit 1930 is momenteel 96 jaar oud. Deze aanname klopt voor de bovenstaande steekproef van 10 personen. 
- Verder is de verdeling van de leeftijden rechtscheef. Het gemiddelde is groter dan de mediaan.

In [ ]:
import unicodedata
def plaats_sleutel(x):
    if leeg(x):
        return None
    s = re.sub(r"\(.*?\)", " ", str(x).lower())          # "Ommen (Ov.)" -> "ommen"
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s or None

persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats"].map(plaats_sleutel)
print(persoon["geboorteplaats_sleutel"].notna().sum(), "ingevuld;", persoon["geboorteplaats_sleutel"].nunique(), "unieke plaatsen")
print(persoon["geboorteplaats_sleutel"].value_counts().head(25).to_string())

Omdat Den Haag ook wel 's gravenhage wordt genoemd of 's hertogenbosch Den Bosch. worden de aliasen samengevoegd.

In [ ]:
ALIAS = {"den haag": "s gravenhage", "den bosch": "s hertogenbosch",
         "hertogenbosch": "s hertogenbosch", "gravenhage": "s gravenhage",}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS)

v = persoon["geboorteplaats_sleutel"].dropna()
for w in ["bosch", "haag", "bergen", "rijswijk", "beek"]:
    print(w, v[v.str.contains(w)].value_counts().to_dict())

vc = v.value_counts()
print(vc.head(10).to_string())
print((vc == 1).sum(), "plaatsen komen maar één keer voor")
print(vc[vc == 1].sample(40, random_state=1).index.tolist())

Lookup tabel gebruiken van CBS om woonplaatsen op gemeente en provincies in te delen.

In [ ]:
import cbsodata
lk = pd.DataFrame(cbsodata.get_data("86312NED"))
print(lk.columns.tolist())
print(lk.head())

In [ ]:
lk = lk.rename(columns={"Woonplaatsen": "woonplaats", "Naam_2": "gemeente",
                        "Code_3": "gemeentecode", "Naam_4": "provincie"})
lk = lk[["woonplaats", "gemeente", "gemeentecode", "provincie"]].copy()
for c in lk.columns:
    lk[c] = lk[c].astype(str).str.strip()

# gemeentenamen ook als zoeksleutel (voor gevallen als "zaanstad")
gem = lk[["gemeente", "gemeentecode", "provincie"]].drop_duplicates().assign(woonplaats=lambda d: d["gemeente"])
lk_all = pd.concat([lk, gem[lk.columns]]).drop_duplicates()

ALIAS2 = {"s heerenbergh": "s heerenberg", "driebergen": "driebergen rijsenburg"}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS2)
LIAS3 = {"capelle a d ijssel": "capelle aan den ijssel", "alphen a d rijn": "alphen aan den rijn",
          "willemstad n b": "willemstad", "cuijk en sint agatha": "cuijk", "cuijk en sint agata": "cuijk",
          "roosendaal en nispen": "roosendaal", "ginneken en bavel": "breda",
          "zuilen": "utrecht", "hillegersberg": "rotterdam", "jutphaas": "nieuwegein",
          "hoogezand sappemeer": "hoogezand", "beemster": "middenbeemster", "dantumadeel": "dantumadiel","mijmegen": "nijmegen", "wonseradeel": "sudwest fryslan", "wymbritseradeel": "sudwest fryslan",
          "dongeradeel": "noardeast fryslan", "westdongeradeel": "noardeast fryslan",
          "doniawerstal": "de fryske marren"}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS3)
lk_all["sleutel"] = lk_all["woonplaats"].map(plaats_sleutel)
n_gem = lk_all.groupby("sleutel")["gemeentecode"].nunique()
eenduidig = (lk_all[lk_all["sleutel"].isin(n_gem[n_gem == 1].index)]
             .drop_duplicates("sleutel")[["sleutel", "gemeente", "gemeentecode", "provincie"]])
ambigu = set(n_gem[n_gem > 1].index)

p = persoon[["id", "geboorteplaats_sleutel"]].merge(
        eenduidig, left_on="geboorteplaats_sleutel", right_on="sleutel", how="left")
assert len(p) == len(persoon)

def status(r):
    if pd.isna(r["geboorteplaats_sleutel"]): return "geen plaats ingevuld"
    if pd.notna(r["gemeente"]):               return "gematcht (eenduidig)"
    if r["geboorteplaats_sleutel"] in ambigu: return "niet eenduidig (meerdere gemeenten)"
    return "niet gevonden"

p["plaats_status"] = p.apply(status, axis=1)
print(p["plaats_status"].value_counts().to_string())
print("niet eenduidig:", p.loc[p["plaats_status"] == "niet eenduidig (meerdere gemeenten)", "geboorteplaats_sleutel"].value_counts().head(15).to_dict())
print("niet gevonden:", p.loc[p["plaats_status"] == "niet gevonden", "geboorteplaats_sleutel"].value_counts().head(40).to_dict())

In [ ]:
ALIAS3 = {"capelle a d ijssel": "capelle aan den ijssel", "alphen a d rijn": "alphen aan den rijn",
          "willemstad n b": "willemstad", "cuijk en sint agatha": "cuijk", "cuijk en sint agata": "cuijk",
          "roosendaal en nispen": "roosendaal", "ginneken en bavel": "breda",
          "zuilen": "utrecht", "hillegersberg": "rotterdam", "jutphaas": "nieuwegein",
          "hoogezand sappemeer": "hoogezand", "beemster": "middenbeemster", "dantumadeel": "dantumadiel"}
missing = [v for v in ALIAS3.values() if plaats_sleutel(v) not in set(lk_all["sleutel"])]
print("alias-doelen die niet in de lookup staan:", missing)